In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No CUDA GPU detected")

In [ ]:
# # # 1. Install required dependencies first (uncomment if running in Colab/Jupyter)
%pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
# %pip install --no-deps xformers trl peft accelerate bitsandbytes
# %pip install --upgrade --force-reinstall --no-cache-dir --no-deps unsloth unsloth_zoo
# %pip install trl

In [ ]:

import torch
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset
from unsloth import FastLanguageModel

# 6. Export to GGUF for FastAPI Integration

In [ ]:
# YOUR_OPENROUTER_API_KEY

In [ ]:
# 2. Load the Base Model
max_seq_length = 2048 
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

# 3. Add LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Target rank
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

In [ ]:

# 4. Format the JSONL Dataset
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token

In [ ]:


def format_prompts(examples):
    instructions = examples["instruction"]
    inputs       = examples["prompt"]
    outputs      = examples["response"]
    texts = []
    
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        # Must add EOS_TOKEN to prevent infinite generation
        text = alpaca_prompt.format(instruction, input_text, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts }

# Load the custom dataset prepared in the previous step
dataset = load_dataset("json", data_files="formatted_training_invoices.jsonl", split="train")
dataset = dataset.map(format_prompts, batched = True)

In [ ]:
# 5. Execute the Fine-Tuning Process
trainer = SFTTrainer(
    model = model,  
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # Increase this based on your full dataset size
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer.train()

In [ ]:
model.save_pretrained_gguf("ai_cfo_llama3", tokenizer, quantization_method = "q4_k_m")

In [ ]:
# 5. Execute the Fine-Tuning Process
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # Increase this based on your full dataset size
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer.train()